# V2: feed-forward reconstruction (DUSt3R) -> Gaussian Splatting

Replaces COLMAP's classical SfM with DUSt3R -- a pretrained, generalizing neural
network that predicts camera poses and a *dense* point cloud in one forward
pass, instead of iterative feature matching + bundle adjustment. This is the
first genuinely trained/generalizing model in this project (Category C: used
pretrained, unmodified, inference only).

The output feeds into the exact same `train_gaussian_splatting` from V0 --
that trainer was refactored to be backend-agnostic (`ReconstructedScene`) so
it doesn't care whether the geometry came from COLMAP or DUSt3R.

**Runtime > Change runtime type > GPU (T4 is fine)** before running these cells.

**Honest flag before you run this:** the pose-convention conversion below
(DUSt3R camera-to-world -> gsplat's expected world-to-camera) is written
against DUSt3R's documented API but the exact axis convention has NOT been
verified against a real run yet -- if the trained result looks inverted or
broken, that conversion is the first place to check.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU attached -- go to Runtime > Change runtime type > GPU, then re-run.')

In [ ]:
!git clone https://github.com/yusupildan-wq/Scene-Reconstruction.git
!git clone --recursive https://github.com/naver/dust3r.git
%cd dust3r

In [ ]:
# torch is already installed (with CUDA) by Colab -- do not reinstall it.
!pip install -q -r requirements.txt
# pycolmap isn't used by the DUSt3R path itself, but runner.py imports it at
# module level (needed for the COLMAP path) -- importing anything from runner.py
# runs that line regardless, so it has to be installed here too. scikit-image
# is only for the PSNR/SSIM numbers in the validation cell later on.
!pip install -q gsplat scipy opencv-python-headless pycolmap scikit-image

import sys
sys.path.insert(0, '.')  # dust3r package (we're inside the dust3r/ clone)
sys.path.insert(0, '../Scene-Reconstruction/backend')
sys.path.insert(0, '../Scene-Reconstruction/worker')

## Upload your video

Same footage you can reuse from the V0 run if you still have it, or a new one.

In [ ]:
from google.colab import files
uploaded = files.upload()
video_filename = next(iter(uploaded.keys()))
print('Uploaded:', video_filename)

In [ ]:
from pathlib import Path
from app.pipeline import extract_frames

frames_dir = Path('real_video_frames')
result = extract_frames(Path(video_filename), frames_dir)
print(f'{result.total_frames_seen} frames seen, {result.selected_frame_count} selected after blur/redundancy filtering')
frame_paths = sorted(str(p) for p in frames_dir.glob('*.jpg'))

## Run DUSt3R: predict poses + dense point cloud in one forward pass

No iterative bundle adjustment here. Pairing uses a **sliding window** (`swin-3`:
each frame pairs with its ~3 nearest neighbors in the sequence) instead of every
possible pair -- our frames come from a video, so nearby frames overlap heavily
and far-apart frames barely overlap at all; pairing everything with everything
(`scene_graph='complete'`) both wastes compute on uninformative pairs and, at 32
images, produced 992 pairs that exhausted Colab's RAM. `inference` runs the
actual forward passes, and `global_aligner` does a comparatively lightweight
optimization to stitch all the pairwise predictions into one consistent scene --
much cheaper than COLMAP's full bundle adjustment.

In [ ]:
from dust3r.inference import inference
from dust3r.model import AsymmetricCroCo3DStereo
from dust3r.utils.image import load_images
from dust3r.image_pairs import make_pairs
from dust3r.cloud_opt import global_aligner, GlobalAlignerMode

device = 'cuda'
model = AsymmetricCroCo3DStereo.from_pretrained(
    'naver/DUSt3R_ViTLarge_BaseDecoder_512_dpt'
).to(device)

images = load_images(frame_paths, size=512)
# swin-3: each frame pairs with ~3 neighbors in sequence order, not every other
# frame -- for 32 images this is on the order of ~100-200 pairs instead of 992,
# and is a better fit for sequential video anyway (see markdown above).
pairs = make_pairs(images, scene_graph='swin-3', prefilter=None, symmetrize=True)
print(f'{len(pairs)} image pairs (was 992 with scene_graph=\"complete\")')
output = inference(pairs, model, device, batch_size=1)

scene = global_aligner(output, device=device, mode=GlobalAlignerMode.PointCloudOptimizer)
loss = scene.compute_global_alignment(init='mst', niter=300, schedule='cosine', lr=0.01)
print('Global alignment final loss:', loss)

## Convert DUSt3R's output into our ReconstructedScene format

This is the actual bridge between DUSt3R and our existing (already-proven)
Gaussian Splatting trainer -- everything after this cell is identical to the
V0 pipeline, just fed denser, feed-forward-predicted geometry instead of
COLMAP's sparser output.

In [ ]:
import numpy as np

def to_numpy(x):
    return x.detach().cpu().numpy() if hasattr(x, 'detach') else np.asarray(x)

masks = scene.get_masks()
pts3d = scene.get_pts3d()
imgs = scene.imgs
poses = scene.get_im_poses()      # camera-to-world
intrinsics = scene.get_intrinsics()

MAX_POINTS_PER_VIEW = 5000  # DUSt3R gives one point per pixel (~200k+ per view) -- way
# denser than we need or than gsplat can handle at once; subsample after confidence filtering.

all_points, all_colors = [], []
for pts, mask, img in zip(pts3d, masks, imgs):
    pts_np = to_numpy(pts).reshape(-1, 3)
    mask_np = to_numpy(mask).reshape(-1)
    img_np = to_numpy(img).reshape(-1, 3)
    valid_pts = pts_np[mask_np]
    valid_colors = img_np[mask_np]
    if len(valid_pts) > MAX_POINTS_PER_VIEW:
        idx = np.random.choice(len(valid_pts), MAX_POINTS_PER_VIEW, replace=False)
        valid_pts, valid_colors = valid_pts[idx], valid_colors[idx]
    all_points.append(valid_pts)
    all_colors.append(valid_colors)

points_xyz = np.concatenate(all_points).astype(np.float32)
points_rgb = np.concatenate(all_colors).astype(np.float32)
print(f'{points_xyz.shape[0]} points from DUSt3R (COLMAP gave us ~1400-7000 on this same kind of footage)')
# (Initial point colors above still come from DUSt3R's 512px images -- fine,
# since this is only a starting guess that training refines away. The camera
# training-supervision images below are a different story -- see next cell.)

## Fix: train against the ORIGINAL video frames, not DUSt3R's 512px preview

DUSt3R resizes every frame so its long edge is 512px before inference --
reasonable for a network whose only job is estimating pose/geometry. The bug
was that `scene.imgs` (those same resized copies) were also being used as the
photometric *training target* for Gaussian Splatting below. That silently
capped this project's entire achievable visual detail at 512px, no matter how
many Gaussians or training iterations get thrown at it -- this is the primary
reason renders have looked soft/grainy rather than sharp, independent of
Gaussian count, training length, or the browser renderer.

The fix: keep DUSt3R for what it's good at (pose + coarse geometry from cheap
low-res inference), but reload each frame at its *original* resolution and
rescale DUSt3R's estimated camera intrinsics (K) to match. `dust3r_crop_region`
below reproduces `dust3r.utils.image.load_images()`'s exact resize+crop math
(verified against DUSt3R's actual source) so the rescaled K stays perfectly
consistent with the poses DUSt3R estimated -- get this wrong and training
would supervise against the right pixels in the wrong place, which would make
results worse, not better.

In [ ]:
from PIL import Image
from PIL.ImageOps import exif_transpose

TARGET_LONG_EDGE = 1024  # was effectively 512 before this fix -- ~4x the pixels,
# still comfortably fits a free-tier T4's memory/time budget. Raise this if you
# have more GPU headroom (paid Colab tier, bigger GPU); each step gets slower
# roughly with pixel count, not just linearly with this number.
PATCH_SIZE = 16  # must match load_images()'s default patch_size

def dust3r_crop_region(orig_w, orig_h, dust3r_size=512, patch_size=PATCH_SIZE):
    """Reproduces dust3r.utils.image.load_images()'s resize+crop exactly, so
    DUSt3R's estimated K (calibrated for its resized+cropped image) can be
    mapped onto pixel coordinates in the ORIGINAL frame instead."""
    scale = dust3r_size / max(orig_w, orig_h)
    W, H = round(orig_w * scale), round(orig_h * scale)
    cx, cy = W // 2, H // 2
    halfw = ((2 * cx) // patch_size) * patch_size / 2
    halfh = ((2 * cy) // patch_size) * patch_size / 2
    if W == H:  # load_images()'s square_ok=False default
        halfh = 3 * halfw / 4
    crop_in_resized_coords = (cx - halfw, cy - halfh, cx + halfw, cy + halfh)
    crop_in_original_coords = tuple(c / scale for c in crop_in_resized_coords)
    dust3r_processed_size = (2 * halfw, 2 * halfh)  # what scene.get_intrinsics() is calibrated for
    return crop_in_original_coords, dust3r_processed_size

camera_viewmats, camera_Ks, camera_images = [], [], []
for i in range(len(imgs)):
    c2w = to_numpy(poses[i])
    # ASSUMPTION flagged in the intro markdown: DUSt3R's camera-to-world matrix
    # is in the same axis convention our gsplat integration already uses
    # successfully (OpenCV/COLMAP-style), so inverting is the only conversion
    # needed. If the trained render looks inverted/broken, revisit this line.
    w2c = np.linalg.inv(c2w).astype(np.float32)
    camera_viewmats.append(w2c)

    K_dust3r = to_numpy(intrinsics[i]).astype(np.float32)
    orig_img = exif_transpose(Image.open(frame_paths[i])).convert('RGB')
    orig_w, orig_h = orig_img.size

    crop_box, processed_size = dust3r_crop_region(orig_w, orig_h)
    cropped = orig_img.crop(tuple(round(c) for c in crop_box))

    resize_scale = min(1.0, TARGET_LONG_EDGE / max(cropped.size))
    final_size = (round(cropped.size[0] * resize_scale), round(cropped.size[1] * resize_scale))
    final_img = cropped.resize(final_size, Image.LANCZOS)
    camera_images.append(np.asarray(final_img, dtype=np.float32) / 255.0)

    # K_dust3r is calibrated for `processed_size` -- rescale to `final_size`,
    # the same field of view/crop, just at our chosen training resolution.
    kx = final_size[0] / processed_size[0]
    ky = final_size[1] / processed_size[1]
    K_scaled = K_dust3r.copy()
    K_scaled[0, 0] *= kx  # fx
    K_scaled[0, 2] *= kx  # cx
    K_scaled[1, 1] *= ky  # fy
    K_scaled[1, 2] *= ky  # cy
    camera_Ks.append(K_scaled)

print('viewmat shape:', camera_viewmats[0].shape, ' K shape:', camera_Ks[0].shape,
      ' image shape:', camera_images[0].shape, ' (was roughly 384x512 at DUSt3R\'s native res before this fix)')

In [ ]:
from runner import ReconstructedScene, train_gaussian_splatting

recon_scene = ReconstructedScene(
    points_xyz=points_xyz,
    points_rgb=points_rgb,
    camera_viewmats=camera_viewmats,
    camera_Ks=camera_Ks,
    camera_images=camera_images,
)

# Denser starting point cloud than COLMAP gave us -- may need less aggressive
# densification to reach good coverage. Same iteration budget as the V0 20k run
# for a fair comparison -- but each iteration is now rasterizing/backpropagating
# through ~4x more pixels per image (1024px vs the previous 512px), so expect
# this run to take noticeably longer in wall-clock time than earlier ones,
# even though the iteration count didn't change.
gaussians = train_gaussian_splatting(recon_scene, num_iterations=20000, densify_until=16000)
print('Trained', gaussians['means'].shape[0], 'Gaussians')

## Validate: does the higher-resolution fix actually show up in the render?

Not "does the point cloud look like a room" -- a direct A/B comparison at one
known camera pose: the real frame we just trained against (A) vs. what the
trained Gaussians render from that exact same pose (B), server-side, via the
same `gsplat` rasterizer used during training. If A and B still look far
apart, the bottleneck wasn't (only) resolution and more training/Gaussians
alone won't fix it -- if they're close, the fix worked and the next thing to
check is the *browser* render against this same B (a separate, already-fixed
concern from earlier in this session).

In [ ]:
import torch
import matplotlib.pyplot as plt
from gsplat import rasterization
from skimage.metrics import structural_similarity, peak_signal_noise_ratio

device = torch.device('cuda')
check_idx = len(camera_images) // 2  # a middle frame, not the first/last (often weakest coverage)

means_t = torch.tensor(gaussians['means'], device=device)
quats_t = torch.tensor(gaussians['quats'], device=device)
scales_t = torch.tensor(gaussians['scales'], device=device)
opacities_t = torch.tensor(gaussians['opacities'], device=device)
colors_t = torch.tensor(gaussians['colors'], device=device)
viewmat_t = torch.tensor(camera_viewmats[check_idx], device=device).unsqueeze(0)
K_t = torch.tensor(camera_Ks[check_idx], device=device).unsqueeze(0)
h, w = camera_images[check_idx].shape[:2]

with torch.no_grad():
    render, _alpha, _meta = rasterization(
        means_t, quats_t, scales_t, opacities_t, colors_t, viewmat_t, K_t, w, h, sh_degree=None
    )

real_frame = camera_images[check_idx]                       # A: what we trained against, at the fixed resolution
rendered_frame = render[0].clamp(0, 1).cpu().numpy()         # B: server-side gsplat render, same pose

psnr = peak_signal_noise_ratio(real_frame, rendered_frame, data_range=1.0)
ssim = structural_similarity(real_frame, rendered_frame, data_range=1.0, channel_axis=2)
print(f'PSNR: {psnr:.2f} dB   SSIM: {ssim:.4f}   (higher is better for both; these support the'
      ' visual comparison below, they do not replace it)')

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(real_frame); axes[0].set_title(f'A: real frame ({w}x{h})'); axes[0].axis('off')
axes[1].imshow(rendered_frame); axes[1].set_title('B: server-side gsplat render, same pose'); axes[1].axis('off')
plt.show()

In [ ]:
import struct
import numpy as np

# 2M+ Gaussians as JSON text would be hundreds of MB and slow for the browser
# to parse -- export as compact raw binary instead (a Float32Array can read
# this directly, zero parsing cost), and cap the count for real-time viewer
# performance (re-sorting millions of points every frame for correct
# transparency blending is real, noticeable overhead in JavaScript).
MAX_VIEWER_POINTS = 500_000

n = gaussians['means'].shape[0]
if n > MAX_VIEWER_POINTS:
    idx = np.random.choice(n, MAX_VIEWER_POINTS, replace=False)
else:
    idx = np.arange(n)

means = gaussians['means'][idx].astype(np.float32)
quats = gaussians['quats'][idx].astype(np.float32)
scales = gaussians['scales'][idx].astype(np.float32)
opacities = gaussians['opacities'][idx].astype(np.float32)
colors = gaussians['colors'][idx].astype(np.float32)

with open('dust3r_scene.bin', 'wb') as f:
    f.write(struct.pack('<I', len(idx)))
    f.write(means.tobytes())
    f.write(quats.tobytes())
    f.write(scales.tobytes())
    f.write(opacities.tobytes())
    f.write(colors.tobytes())

import os
size_mb = os.path.getsize('dust3r_scene.bin') / 1e6
print(f'Exported {len(idx)} / {n} Gaussians as binary ({size_mb:.1f} MB)')

files.download('dust3r_scene.bin')